In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("=== PARTIE I : MLP ET INGÉNIERIE PYTORCH ===")

# ==========================================
# 2. Préparation des données
# ==========================================
print("\n--- 2. Préparation des données ---")
data = load_breast_cancer()
X = data.data
y = data.target

# Séparation: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Normalisation (Fit uniquement sur le train pour éviter le data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Conversion en Tenseurs PyTorch
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # Unsqueeze pour BCELoss
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# ==========================================
# 3. Implémentation des deux versions du MLP
# ==========================================
print("\n--- 3. Construction des modèles ---")
input_dim = X_train.shape[1] # 30 features

# Version 1 : Avec nn.Sequential
mlp_sequential = nn.Sequential(
    nn.Linear(input_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1) # 1 sortie pour classification binaire (BCEWithLogitsLoss)
)

# Version 2 : Avec une classe personnalisée (Celle qu'on va entraîner)
class CustomMLP(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)

model = CustomMLP(input_dim)

# ==========================================
# 4. Inspection des paramètres
# ==========================================
print("\n--- 4. Inspection avec named_parameters() ---")
for name, param in model.named_parameters():
    print(f"Paramètre: {name} | Dimension: {param.shape} | Nécessite un gradient: {param.requires_grad}")

# ==========================================
# 5. Stratégies d'initialisation
# ==========================================
print("\n--- 5. Initialisation des poids ---")
def init_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

def init_constant(m):
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 1.0) # Mauvaise pratique en réalité
        nn.init.zeros_(m.bias)

def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

# On applique l'initialisation de Xavier (recommandée)
model.apply(init_xavier)
print("Initialisation de Xavier appliquée avec succès.")

# ==========================================
# 7. Exécution sur le Device (CPU/GPU) et Entraînement
# ==========================================
print("\n--- 7. Transfert sur le Device et Entraînement ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

# Déplacement du modèle et des données sur le device
model = model.to(device)
X_train_t, y_train_t = X_train_t.to(device), y_train_t.to(device)
X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)
X_test_t, y_test_t = X_test_t.to(device), y_test_t.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # Propagation avant
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    # Rétropropagation
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Époque [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# ==========================================
# 6. Sauvegarde et rechargement
# ==========================================
print("\n--- 6. Sauvegarde et Rechargement du modèle ---")
torch.save(model.state_dict(), "mlp_breast_cancer_weights.pth")
print("Modèle sauvegardé dans 'mlp_breast_cancer_weights.pth'")

# Rechargement dans une nouvelle instance
best_model = CustomMLP(input_dim)
best_model.load_state_dict(torch.load("mlp_breast_cancer_weights.pth", weights_only=True))
best_model = best_model.to(device)
print("Modèle rechargé avec succès.")

# ==========================================
# 8. Évaluation des performances
# ==========================================
print("\n--- 8. Évaluation sur le jeu de Test ---")
best_model.eval()
with torch.no_grad():
    test_logits = best_model(X_test_t)
    # Application de la sigmoïde pour obtenir des probabilités, puis seuillage à 0.5
    test_preds = (torch.sigmoid(test_logits) > 0.5).float()

# Retour sur CPU pour utiliser Scikit-Learn
y_true = y_test_t.cpu().numpy()
y_pred = test_preds.cpu().numpy()

print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_true, y_pred):.4f}")
print("\nMatrice de Confusion :")
print(confusion_matrix(y_true, y_pred))

=== PARTIE I : MLP ET INGÉNIERIE PYTORCH ===

--- 2. Préparation des données ---

--- 3. Construction des modèles ---

--- 4. Inspection avec named_parameters() ---
Paramètre: fc1.weight | Dimension: torch.Size([64, 30]) | Nécessite un gradient: True
Paramètre: fc1.bias | Dimension: torch.Size([64]) | Nécessite un gradient: True
Paramètre: fc2.weight | Dimension: torch.Size([32, 64]) | Nécessite un gradient: True
Paramètre: fc2.bias | Dimension: torch.Size([32]) | Nécessite un gradient: True
Paramètre: out.weight | Dimension: torch.Size([1, 32]) | Nécessite un gradient: True
Paramètre: out.bias | Dimension: torch.Size([1]) | Nécessite un gradient: True

--- 5. Initialisation des poids ---
Initialisation de Xavier appliquée avec succès.

--- 7. Transfert sur le Device et Entraînement ---
Device utilisé : cpu
Époque [20/100], Loss: 0.2437
Époque [40/100], Loss: 0.1217
Époque [60/100], Loss: 0.0853
Époque [80/100], Loss: 0.0681
Époque [100/100], Loss: 0.0569

--- 6. Sauvegarde et Recharge